In [1]:
from utils import pr
from utils import measure_time
import numpy as np
import pandas as pd

In [2]:
# User table (one row per user)
users = pd.DataFrame(
    {
        "user_id": ["001", "002", "003", "004"],
        "signup_date": pd.to_datetime(
            ["2023-01-01", "2023-01-05", "2023-01-10", "2023-01-15"]
        ),
        "churned": [False, True, False, True],
    }
)
# Transactions table (events)
transactions = pd.DataFrame(
    {
        "user_id": ["001", "001", "002", "003", "003", "004"],
        "transaction_date": pd.to_datetime(
            [
                "2023-01-03",
                "2023-02-10",
                "2023-01-06",
                "2023-01-12",
                "2023-03-01",
                "2023-02-20",
            ]
        ),
        "amount": [100, 500, 50, 200, 1000, 300],
    }
)
pr(users , 'users')
pr(transactions , 'transactions')

+++++++++++++++++ users ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_date,churned
0,001,2023-01-01,False
1,002,2023-01-05,True
2,003,2023-01-10,False
3,004,2023-01-15,True


-----

+++++++++++++++++ transactions ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,transaction_date,amount
0,001,2023-01-03,100
1,001,2023-02-10,500
2,002,2023-01-06,50
3,003,2023-01-12,200
4,003,2023-03-01,1000
5,004,2023-02-20,300


-----



You have to keep strong mental separation between Features and labels
- Features (what the model is allowed to see)
- Labels (what the model must predict)

In [9]:
merged = users.merge(transactions , on='user_id' , how='left')
pr(merged , 'row merged row naively joined')
print('We will enforce temporal boundary')
merged = merged[
    merged['transaction_date'] <= merged['signup_date'] + pd.Timedelta(days=7)
]
pr(merged , 'enforced temporal boundary')
pr('then aggregate safely')
features = merged.groupby('user_id').agg(
    spend_7d = pd.NamedAgg('amount' , 'sum') , 
    tx_count_7d = pd.NamedAgg('amount' , 'count')
).reset_index()
pr(features , 'after aggregating safely')

+++++++++++++++++ row merged row naively joined ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_date,churned,transaction_date,amount
0,001,2023-01-01,False,2023-01-03,100
1,001,2023-01-01,False,2023-02-10,500
2,002,2023-01-05,True,2023-01-06,50
3,003,2023-01-10,False,2023-01-12,200
4,003,2023-01-10,False,2023-03-01,1000
5,004,2023-01-15,True,2023-02-20,300


-----

We will enforce temporal boundary
+++++++++++++++++ enforced temporal boundary ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,signup_date,churned,transaction_date,amount
0,001,2023-01-01,False,2023-01-03,100
2,002,2023-01-05,True,2023-01-06,50
3,003,2023-01-10,False,2023-01-12,200


-----

+++++++++++++++++++++++++++++++++++
<class 'str'>


'then aggregate safely'

-----

+++++++++++++++++ after aggregating safely ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,spend_7d,tx_count_7d
0,001,100,1
1,002,50,1
2,003,200,1


-----



In [10]:
labels = users[['user_id' , 'churned']]
pr(labels , 'labels')
training_data = features.merge(labels , on='user_id' , how='left')
pr(training_data)

+++++++++++++++++ labels ++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,churned
0,001,False
1,002,True
2,003,False
3,004,True


-----

+++++++++++++++++++++++++++++++++++
<class 'pandas.DataFrame'>


,user_id,spend_7d,tx_count_7d,churned
0,001,100,1,False
1,002,50,1,True
2,003,200,1,False


-----

